# 01 — Data Extraction & Dataset Split

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** Download images from Pexels and split them into `split_dataset/train/` and `split_dataset/test/`.

All logic lives in `src/data_extraction.py` and `src/split_data.py`.  
This notebook is the single place to run both steps.

---
### Folder output after running this notebook
```
AuraVision/
├── crowd_dataset/          ← all downloaded training images
│   ├── Children/
│   ├── Adults/
│   └── Seniors/
├── test_dataset/           ← all downloaded test images
│   ├── Children/
│   ├── Adults/
│   └── Seniors/
└── split_dataset/          ← 80/20 split ready for training
    ├── train/
    └── test/
```

## Step 0 — Setup

In [ ]:
# Install dependencies (run once)
!pip install -q requests python-dotenv

import os, sys

# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AuraVision'
    print('Running in Colab — project root:', PROJECT_ROOT)
except ImportError:
    PROJECT_ROOT = os.path.abspath('..')
    print('Running locally — project root:', PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Working directory:', os.getcwd())

## Step 1 — Configure API Key

Your Pexels API key is stored in `.env` — **never hardcoded**.  
If the file does not exist yet, create it:
```
PEXELS_API_KEY=your_actual_key_here
```

In [ ]:
from dotenv import load_dotenv
load_dotenv()   # loads .env from the project root

api_key = os.getenv('PEXELS_API_KEY')
if not api_key or api_key == 'your_pexels_api_key_here':
    raise EnvironmentError(
        'PEXELS_API_KEY not set!\n'
        'Add it to AuraVision/.env:\n'
        '  PEXELS_API_KEY=your_actual_key_here'
    )
print('✅ API key loaded successfully.')

## Step 2 — Download Training Images

Downloads images for **Children**, **Adults**, and **Seniors** into `crowd_dataset/`.  
~40 images per query × 6-8 queries per class ≈ 240–320 images per class.

In [ ]:
from data_extraction import download_train_images
download_train_images()

## Step 3 — Download Test Images

Downloads a separate, more context-varied set into `test_dataset/`.  
Queries are intentionally different from training to test generalisation.

In [ ]:
from data_extraction import download_test_images
download_test_images()

## Step 4 — Split Into Train / Test

Shuffles `crowd_dataset/` and copies 80% → `split_dataset/train/`, 20% → `split_dataset/test/`.  
The split is reproducible (seed = 42).

In [ ]:
from split_data import split_dataset
split_dataset(
    source_dir='crowd_dataset',
    output_dir='split_dataset',
    split_ratio=0.8,
)

## Step 5 — Verify Dataset

Count images in each split to confirm everything looks right.

In [ ]:
for subset in ('train', 'test'):
    print(f'\n── split_dataset/{subset}/')
    base = os.path.join('split_dataset', subset)
    for cls in sorted(os.listdir(base)):
        cls_path = os.path.join(base, cls)
        if os.path.isdir(cls_path):
            count = len([f for f in os.listdir(cls_path) if f.endswith('.jpg')])
            print(f'  {cls:<12} {count} images')